In [53]:
import os

from dotenv import load_dotenv
from training.common import FeedID, Operator
from training.data_processing import feed_message_to_vehicle_position_dataframe, join_static_data_on_rt_vehicle_positions
from training.gtfs import load_gtfs_rt_immediately
from training.static_data import StaticData
load_dotenv()


folder_path = "../../../../data"
model_path = "../../../../models"


GTFS_RT_API_KEY = os.environ.get("GTFS_REGIONAL_RT_API_KEY", "")
if GTFS_RT_API_KEY == "":
    raise Exception("GTFS_REGIONAL_RT_API_KEY environment variable not set")

gtfs_static_data = StaticData.load_static_data(f"{folder_path}/gtfs-static/data-tmp")

STAM_BUSES_ROUTE_SET = {"9011001000100000", 
                    "9011001000200000",
                    "9011001000300000",
                    "9011001000400000"}

In [ ]:

import datetime
from training.gtfs import list_gtfs_files, load_gtfs_frame_from_files
from training.data_pipeline import lag_times
from training.data_processing import explode_to_stops_with_join_static, join_static_data_on_rt_trip_updates
from training.model_training import load_route_model
import pandas as pd
from training.data_pipeline import create_X_from_df
from xgboost import XGBRegressor

def do_inference_on_route(route_id: str, model: XGBRegressor, data: pd.DataFrame) -> pd.DataFrame:
    filtered_data = data[data["route_id"] == route_id].copy()
    filtered_data["arrival_time_late_prev"] = filtered_data["arrival_time_late_prev"].fillna(
        pd.to_timedelta(1, unit="s")
    )

    x_data = create_X_from_df(filtered_data)

    late_preds = model.predict(x_data)

    # ensure datetime type + add timedelta seconds
    filtered_data["arrival_time_estimate"] = (
        pd.to_datetime(filtered_data["arrival_time_planned"])
        + pd.to_timedelta(pd.Series(late_preds), unit="s")
    )

    return filtered_data

xgbModels:dict[str, XGBRegressor] = {}

for route in STAM_BUSES_ROUTE_SET:
    xgbModels[route] = load_route_model(route, model_path=model_path)

gtfs_vehicle_positions = load_gtfs_rt_immediately(
    operator=Operator.SL, 
    feedId=FeedID.VehiclePositions, 
    api_key= GTFS_RT_API_KEY,
)

buses_df = feed_message_to_vehicle_position_dataframe(gtfs_vehicle_positions)
buses_df = join_static_data_on_rt_vehicle_positions(gtfs_static_data, buses_df)

now = datetime.datetime.now(datetime.timezone.utc)
ten_minute_files = list_gtfs_files(
    base_path=f"{folder_path}/gtfs-rt/data-tmp/sl/TripUpdates",
    start_dt=now - datetime.timedelta(minutes=10),
    end_dt=now
)

gtfs_feed_df = load_gtfs_frame_from_files(ten_minute_files, gtfs_static_data)

trip_updates_df = gtfs_feed_df[gtfs_feed_df["route_id"].isin(STAM_BUSES_ROUTE_SET)]
trip_updates_df = join_static_data_on_rt_trip_updates(gtfs_static_data, trip_updates_df)
trip_updates_df = explode_to_stops_with_join_static(gtfs_static_data, trip_updates_df)
trip_updates_df = lag_times(trip_updates_df)


# Only keep the last line stop_sequence per trip_id
# multiindex: trip_id	stop_sequence	
trip_updates_df = trip_updates_df.sort_values(['trip_id', 'stop_sequence'])
trip_updates_df = trip_updates_df.groupby(level='trip_id').last()

trip_updates_df = trip_updates_df.reset_index()


inferences = []

for route_id in STAM_BUSES_ROUTE_SET:
    inferences.append(do_inference_on_route(route_id=route_id, model=xgbModels[route_id], data=trip_updates_df))

inference_df = pd.concat(inferences)
buses_df = buses_df.set_index("trip_id")
inference_df = inference_df.reset_index()
inference_df = inference_df.set_index("trip_id")

inference_df = inference_df.join(buses_df, on="trip_id", rsuffix="_trip", how="inner")

inference_df = inference_df.reset_index()

inference_df.head()

arrival_time            datetime64[ns]
arrival_time_planned    datetime64[ns]
dtype: object


,index,id,start_date,schedule_relationship,vehicle_id,timestamp,route_id,service_id,trip_headsign,direction_id,...,route_id_trip,service_id_trip,trip_headsign_trip,direction_id_trip,shape_id_trip,agency_id_trip,route_short_name_trip,route_long_name_trip,route_type_trip,route_desc_trip
trip_id,,,,,,,,,,,,,,,,,,,,,
14010000664229038,1,14010517577208696,2026-01-12,0,9031001001007162,1768218230,9011001000300000,12,<NA>,1.0,...,9011001000300000,12,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss
14010000664229229,2,14010517577200179,2026-01-12,0,9031001001001555,1768218230,9011001000300000,12,<NA>,1.0,...,9011001000300000,12,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss
14010000664229564,3,14010517577211909,2026-01-12,0,9031001001001549,1768218230,9011001000300000,12,<NA>,1.0,...,9011001000300000,12,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss
14010000664229758,4,14010517577203596,2026-01-12,0,9031001001007161,1768218230,9011001000300000,12,<NA>,1.0,...,9011001000300000,12,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss
14010000702275795,8,14010517576957480,2026-01-12,0,9031001001007203,1768218230,9011001000300000,12,<NA>,0.0,...,9011001000300000,12,<NA>,0.0,1014010000573107346,14010000000001001,3,<NA>,700,blåbuss


In [55]:
inference_df[["vehicle_id", "route_short_name", "stop_id", "arrival_time_planned", "arrival_time_estimate",
            
            "arrival_time_planned_prev",
            "arrival_time_late_prev",]]

,vehicle_id,route_short_name,stop_id,arrival_time_planned,arrival_time_estimate,arrival_time_planned_prev,arrival_time_late_prev
trip_id,,,,,,,
14010000664229038,9031001001007162,3,9022001010406001,2026-01-12 12:51:00,2026-01-12 12:51:49.592716217,2026-01-12 12:44:12,0 days 00:02:28
14010000664229229,9031001001001555,3,9022001010406001,2026-01-12 13:01:00,2026-01-12 13:14:18.758544922,2026-01-12 12:54:12,0 days 00:16:41
14010000664229564,9031001001001549,3,9022001010406001,2026-01-12 13:11:00,2026-01-12 13:12:47.801414490,2026-01-12 13:04:12,0 days 00:05:22
14010000664229758,9031001001007161,3,9022001010406001,2026-01-12 13:21:00,2026-01-12 13:23:27.062667847,2026-01-12 13:14:12,0 days 00:04:05
14010000702275795,9031001001007203,3,9022001050800003,2026-01-12 12:40:00,2026-01-12 12:40:26.526292801,2026-01-12 12:31:55,0 days 00:10:05
14010000702276053,9031001001007177,3,9022001050800003,2026-01-12 12:50:00,2026-01-12 12:50:37.852596283,2026-01-12 12:41:55,0 days 00:07:47
14010000702276204,9031001001007157,3,9022001050800003,2026-01-12 13:01:00,2026-01-12 13:00:57.861336708,2026-01-12 12:52:55,0 days 00:07:09
14010000702276417,9031001001001550,3,9022001050800003,2026-01-12 13:11:00,NaT,2026-01-12 13:02:55,0 days 00:04:12
14010000702276598,9031001001001558,3,9022001050800003,2026-01-12 13:21:00,NaT,2026-01-12 13:12:55,0 days 00:02:20


In [64]:
SL_AGENCY_ID = gtfs_static_data.agencies[gtfs_static_data.agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]

inference_df = inference_df.reset_index()
buses = inference_df[(inference_df["route_short_name"] == "1") & (inference_df["agency_id"] == SL_AGENCY_ID)]

        # id=str(row["id"]),
        # trip_id=row["trip_id"],
        # position=VehiclePosition(
        #     latitude=row["vehicle_latitude"],
        #     longitude=row["vehicle_longitude"],
        #     bearing=row["vehicle_bearing"],
        #     speed=row["vehicle_speed"],
        #     odometer=row["vehicle_odometer"],
        # ),
        # next_stop_id=row["stop_id"],
        # next_stop_scheduled_arrival_time=row["arrival_time_planned"],
        # next_stop_estimated_arrival_time=row["arrival_time_estimate"],
buses[["id", "trip_id", "vehicle_latitude", "vehicle_longitude", "vehicle_bearing", "vehicle_speed", "vehicle_odometer", "stop_id", "arrival_time_planned", "arrival_time_estimate"]]

,id,trip_id,vehicle_latitude,vehicle_longitude,vehicle_bearing,vehicle_speed,vehicle_odometer,stop_id,arrival_time_planned,arrival_time_estimate
10,14010517574830379,14010000669012473,59.324009,17.994774,323.0,2.5,0.0,9022001010736002,2026-01-12 12:37:00,2026-01-12 12:49:17.776184082
11,14010517574830442,14010000694062320,59.340672,18.115351,52.0,8.9,0.0,9022001010028001,2026-01-12 12:40:00,2026-01-12 12:40:51.921134949
12,14010517578563624,14010000698528268,59.333843,18.037165,106.0,7.8,0.0,9022001010028001,2026-01-12 13:10:00,2026-01-12 13:12:26.978027344
13,14010517578563120,14010100672069915,59.325146,18.001301,229.0,9.7,0.0,9022001010736002,2026-01-12 12:47:00,NaT
14,14010517574830568,14010100694062520,59.340221,18.075975,27.0,0.0,0.0,9022001010028001,2026-01-12 12:50:00,NaT
15,14010517574830631,14010100694062846,59.340199,18.077250,111.0,3.9,0.0,9022001010028001,2026-01-12 13:00:00,NaT
16,14010517574830820,14010100694063375,59.328869,18.021936,16.0,8.1,0.0,9022001010028001,2026-01-12 13:20:00,NaT
17,14010517578562868,14010100698527906,59.341110,18.117640,64.0,0.0,0.0,9022001010028001,2026-01-12 12:30:00,NaT
18,14010517574830757,14010100702343791,59.335026,18.061405,244.0,0.6,0.0,9022001010736002,2026-01-12 13:08:00,NaT
19,14010517578563372,14010100708559249,59.334660,18.031006,283.0,0.0,0.0,9022001010736002,2026-01-12 12:58:00,NaT


In [ ]:

from pydantic import BaseModel


class VehiclePosition(BaseModel):
    latitude: float
    longitude: float
    bearing: float
    speed: float
    odometer: float

class Vehicle(BaseModel):
    id: str
    position: VehiclePosition
    trip_id: str
    next_stop_id: str
    next_stop_scheduled_arrival_time: float
    next_stop_estimated_arrival_time: float | None

def nan_if_nat(value: pd.Timestamp) -> float | None:
    if pd.isnull(value):
        return None
    
    return value.timestamp()

def bus_row_to_vehicle(row) -> Vehicle:
    return Vehicle(
        id=str(row["id"]),
        trip_id=row["trip_id"],
        position=VehiclePosition(
            latitude=row["vehicle_latitude"],
            longitude=row["vehicle_longitude"],
            bearing=row["vehicle_bearing"],
            speed=row["vehicle_speed"],
            odometer=row["vehicle_odometer"],
        ),
        next_stop_id=row["stop_id"],
        next_stop_scheduled_arrival_time=row["arrival_time_planned"].timestamp(),
        next_stop_estimated_arrival_time=nan_if_nat(row["arrival_time_estimate"]),
    )


result_vehicles = [bus_row_to_vehicle(row) for _, row in buses.iterrows()]

result_vehicles


ValueError: NaTType does not support timestamp